In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from eucare.base import *
import eucare.plotting as euplot


plt.figure(figsize=(10, 10))
prev_poly = regular_poly_points(3)
for i in range(3, 20):
    poly = regular_poly_points(i) * np.random.rand()
    j = np.random.randint(0, i-2)
    mat = find_affine(poly[j:j+2][::-1], prev_poly[i-3:i-1])
    poly = apply_affine(poly, mat)
    euplot.plot_polygon(poly)
    prev_poly=poly
    #plt.scatter(*(poly[j:j+2].T))
    
euplot.set_equal_aspect()

In [ ]:
np.linspace(0, 10, 10, endpoint=False)

In [ ]:
nearest_neighbor(np.random.rand(100, 2), np.random.rand(2))

In [ ]:
from networkx import DiGraph
graph = DiGraph()
#graph.add_nodes_from([1, 2, 3])
graph.add_edges_from([[1, 2], [3, 2], [3, 1]])


In [ ]:
from eucare.graph import NEFGraph
import networkx as nx

def square_tiling():
    tiling = NEFGraph()
    for i in range(10):
        for j in range(10):
            tiling.add_face([(i, j), (i+1, j), (i+1, j+1), (i, j+1)])
    return tiling
#%timeit tiling = square_tiling()
tiling = square_tiling()
print(len(tiling.nodes))
print(len(tiling.inner_nodes))
#[print(x) for x in tiling.n2f_cycle((2, 2))]
#print(len(tiling.nef.edges))
#print(tiling.border_edges)
%timeit tiling.dual()
#%timeit tiling.n2f_cycle((15, 7))
assert False
G = tiling.construct_n2n_graph()#(include_dangling=False)
plt.figure(figsize=(15, 15))
pos = nx.spring_layout(G.to_undirected())
nx.draw_networkx_nodes(G, pos, cmap=plt.get_cmap('jet'), node_size = 500)
nx.draw_networkx_edges(G, pos, edge_color='r', arrows=True)
nx.draw_networkx_labels(G, pos);

#print(tiling._cycle_around(tiling.faces[0]))

In [ ]:
200e-6 * 2500

## HalfEdge Data Structure

In [ ]:
from itertools import chain, tee
import networkx as nx

class IdObject:
    current_ids = dict()
    def __init__(self):
        super(IdObject, self).__init__()
        cls = type(self)
        IdObject.current_ids[cls] = IdObject.current_ids.get(cls, 0) + 1
        self.id = IdObject.current_ids[cls]
        
    def __repr__(self):
        cls = type(self)
        clspre = cls.printname if hasattr(cls, 'printname') else cls.__name__
        return f'{clspre}{self.id}'
    
    @classmethod
    def reset_ids(cls):
        if cls is IdObject:
            print('resetting all ids')
            IdObject.current_ids = dict()
        else:
            IdObject.current_ids[cls] = 0

                   
class HalfEdge(IdObject):
    printname = 'HE'
    
    def __init__(self, rev=None, nex=None, pre=None, orig=None, dest=None, face=None):
        super(HalfEdge, self).__init__()
        # HalfEdge references
        self.rev = rev
        self.nex = nex
        self.pre = pre
        
        # Vertex references
        self.orig = orig
        self.dest = dest
        
        # Face reference
        self.face = None
    
    def on_border(self):
        return self.face is None
                   
    
# class Edge(tuple):
#     def __new__(cls, halfedge0, halfedge1):
#         if not (halfedge0.orig == halfedge1.dest and 
#                 halfedge0.dest == halfedge1.orig and
#                 halfedge0.rev == halfedge1 and
#                 halfedge1.rev == halfedge0):
#             raise ValueError(f"Halfedges ({halfedge0}, {halfedge1}) not consistent.")
#         return super(Edge, cls).__new__(cls, (halfedge0, halfedge1))
        
#     def __getitem__(self, idx):
#         if isinstance(idx, int):
#             # if idx is an int, treat normally
#             return super(Edge, self).__getitem__(idx)
#         else:
#             # idx should be one of the vertices
#             for i in (0, 1):
#                 if idx == super(Edge, self).__getitem__(i).orig:
#                     return super(Edge, self).__getitem__(i)
#                 raise ValueError(f"Vertex {idx} not found")


class Vertex(IdObject):
    printname = 'V'
    def __init__(self, any_outgoing=None):
        super(Vertex, self).__init__()
        self.any_outgoing = any_outgoing
               
    def outgoing_iter(self):
        initial = self.any_outgoing
        current = initial
        while True:
            yield current
            current = current.pre.rev
            if current is initial:
                break
                
    def incoming_iter(self):
        for h in self.outgoing_iter():
            yield h.rev
                
    def face_iter(self):
        for h in self.outgoing_iter():
            yield h.face
        
    def vertex_iter(self):
        for h in self.outgoing_iter():
            yield h.dest
            
    def on_border(self):
        return any(h.on_border() for h in self.outgoing_iter)
    
    def get_outgoing_border(self):
        # search for borders
        border_edges = [h for h in self.outgoing_iter() if h.face is None]
        assert len(border_edges) > 0, f'Vertex {self} does not lie on a border.'
        assert len(border_edges) < 2, f'Vertex {self} has multiple adjacent border edges. Please specify one.'
        return border_edges[0]
    
    def combine_with(self, other):
        # This method might be used in subclasses to e.g. average positions when vertices are combined.
        # For now, just use one of them.
        return self

            
class Face:
    def __init__(self, any_side=None):
        self.any_side = any_side
    
    def halfedge_iter(self):
        initial = self.any_side
        current = initial
        while True:
            yield current
            current = current.nex
            if current is initial:
                break
    
    def reverse_halfedge_iter(self):
        for h in self.halfedge_iter():
            yield h.rev
            
    def vertex_iter(self):
        for h in self.halfedge_iter():
            yield h.orig
            
    def face_iter(self):
        for h in self.halfedge_iter():
            yield h.rev.face
            
        
class HalfEdgeGraph:
    def __init__(self, other=None):
        if other is not None:
            if isinstance(other, HalfEdgeGraph):
                self.halfedges = copy(other.halfedges)
                self.vertices = copy(other.vertices)
                self.faces = copy(other.faces)
            else:
                raise NotImplementedError
        else:
            self.halfedges = set()
            self.vertices = set()
            self.faces = set()
            
    def add_graph(self, other):
        if not isinstance(other, HalfEdgeGraph):
            raise TypeError('other must be a HalfEdgeGraph.')
        self.add_halfedges(other.halfedges)
        self.add_vertices(other.vertices)
        self.add_faces(other.faces)
        
    @property
    def order(self):
        return len(self.vertices)
    
    @property
    def size(self):
        return len(self.halfedges) // 2
        
    def add_vertex(self, v):
        self.vertices.add(v)
        
    def add_vertices(self, vs):
        self.vertices.update(vs)
        
    def add_face(self, f):
        self.faces.add(f)
        
    def add_faces(self, fs):
        self.faces.update(fs)
        
    def add_halfedge(self, h):
        self.halfedges.add(h)
        
    def add_halfedges(self, hs):
        self.halfedges.update(hs)
    
    # TODO: handle removal
    
    def to_networkx_undirected(self):
        result = nx.Graph()
        result.add_edges_from([(h.orig, h.dest) for h in self.halfedges])
        return result
#     def add_implicit(self):
#         self.add_vertices(
#             [h.orig for h in self.halfedges] +
#             [h.dest for h in self.halfedges]
#         )
#         self.add_edges

    def get_any_border(self):
        for h in self.halfedges:
            if h.on_border():
                return h
        raise LookupError('No border found.')
        
    def border_edge_iter(self):
        initial = self.get_any_border()
        current = initial
        while True:
            yield current
            current = current.nex
            if current is initial:
                break
                
    def border_edges(self):
        return list(self.border_edge_iter())
                
    def border_vertex_iter(self):
        for h in self.border_edge_iter():
            yield h.orig
            
    def border_vertices(self):
        return list(self.border_vertex_iter())

    def glue_v2v(self, v1=None, v2=None, v1_out=None, v2_out=None):
        # check if both vertices are suited for gluing
        if v1 is None:
            assert v1_out is not None, f'v1 or v1_out must be specified.'
            v1 = v1_out.orig
        if v2 is None:
            assert v2_out is not None, f'v2 or v2_out must be specified.'
            v2 = v2_out.orig
            
        # search for border edges if not specified
        v1_out = v1.get_outgoing_border() if v1_out is None else v1_out
        v2_out = v2.get_outgoing_border() if v2_out is None else v2_out
        assert (v1_out.orig is v1) and (v2_out.orig is v2)
        
        # get incoming border edges
        v1_in, v2_in = v1_out.pre, v2_out.pre
        
        # assign new vertex, remove old
        v = v1.combine_with(v2)
        for h in chain(v1.outgoing_iter(), v2.outgoing_iter()):
            h.orig = v
            h.rev.dest = v
        self.vertices.difference_update({v1, v2})
        self.vertices.add(v1)
            
        # do the shuffle
        v1_out.pre = v2_in
        v2_out.pre = v1_in
        v1_in.nex = v2_out
        v2_in.nex = v1_out
        
        
    def glue_e2e(self, e1, e2):
        for e in (e1, e2):
            if not e.on_border():
                raise ValueError(f'Cannot glue: Edge {e} not on border.')
                
        # glue vertices
        for v1, v2 in ((e1.orig, e2.dest), (e1.dest, e2.orig)):
            if v1 != v2:
                self.glue_v2v(v1, v2)
        
        # eliminate double edge
        e1.rev.rev = e2.rev
        e2.rev.rev = e1.rev
        self.halfedges.difference_update({e1, e2})
        
    def close_vertex(self, v):
        # get edges to be glued
        e1 = v.get_outgoing_border()
        e2 = e1.pre
        self.glue_e2e(e1, e2)

#def rotate_left(iterator_like):
#    iterator = iter(iterator_like)
#    last = next(iterator)
#    for item in iterator:
#        yield item
#    yield last
        
# def rotate_by(iterable, offset):
#     def rotate_by_single(iterable, offset):
#         if isinstance(offset, int):
#             first_n = [next(iterable) for _ in range(offset)]
#             for value in chain(iterable, first_n):
#                 yield value
    
#     if isinstance(offset, int):
#         return rotate_by_single(iterable, offset)
#     else:
#         iterables = tee(iterable, len(offset))
#         return zip(*(rotate_by_single(iterable, off)
#                     for iterable, off in zip(iterables, offset)))
    
    
def rotate_by(list_like, offset):
    if isinstance(offset, int):
        l = list(list_like)
        return l[offset:] + l[:offset]
    else:
        return zip(*(rotate_by(list_like, off) for off in offset))

    
def any_element(s):
    return next(iter(s))
    
#def cyclic_pairs(iterator_like):
#    return zip(iter(iterator_like), rotate_left(iterator_like))


class CyclicHalfedgeGraph(HalfEdgeGraph):
    def __init__(self, vs, f=None):
        super(CyclicHalfedgeGraph, self).__init__()
        
        # init face if necessary 
        f = Face() if f is None else f
        assert isinstance(f, Face), f"{type(f)}"
        
        # init the halfedges, first only with the vertices
        inner_hs = [HalfEdge(orig=orig, dest=dest) 
                          for orig, dest in rotate_by(vs, (0, 1))]
        outer_hs = [HalfEdge(orig=orig, dest=dest) 
                          for dest, orig in rotate_by(vs, (0, 1))]
        
        # give face reference to a halfedge
        f.any_side = inner_hs[0]
        
        # nex and pre and face for inner halfedges
        for h, pre, nex in rotate_by(inner_hs, (0, -1, 1)):
            h.nex = nex
            h.pre = pre
            h.face = f
            h.orig.any_outgoing = h
        
        # nex and pre for outer halfedges
        for h, pre, nex in rotate_by(outer_hs, (0, 1, -1)):  # note different offsets
            h.nex = nex
            h.pre = pre
        
        # rev for all halfedges
        for h_inner, h_outer in zip(inner_hs, outer_hs):
            h_inner.rev = h_outer
            h_outer.rev = h_inner
            
        # finally, add everything to self
        self.add_vertices(vs)
        self.add_face(f)
        self.add_halfedges(inner_hs)
        self.add_halfedges(outer_hs)
        # TODO: add edges

In [ ]:
from tqdm import tqdm_notebook as tqdm

IdObject.reset_ids()
poly = CyclicHalfedgeGraph([Vertex() for i in range(4)])
#for v in poly.vertices:
#    print(v)
#    for h in v.outgoing_iter():
#        pass
#        print(h.orig, h.dest)

#hs = list(poly.halfedges)

#f = any_element(poly.faces)
#print(f.__dict__)
#[print(v) for v in f.reverse_halfedge_iter()]


for i in range(15):
    print(i, poly.order)
    border_vertices = poly.border_vertices()
    for h1 in tqdm(list(poly.border_edge_iter())):
        #print(h1)
        to_attach = CyclicHalfedgeGraph([Vertex() for i in range(np.random.randint(4, 7))])
        h2 = to_attach.get_any_border()
        poly.add_graph(to_attach)
        poly.glue_e2e(h1, h2)
    for v in border_vertices:
        poly.close_vertex(v)
        

G = poly.to_networkx_undirected()
pos = nx.spring_layout(G.to_undirected())
plt.figure(figsize=(15, 15))
nx.draw_networkx_nodes(G, pos, cmap=plt.get_cmap('jet'), node_size = 500)
nx.draw_networkx_edges(G, pos, edge_color='r', arrows=True)
nx.draw_networkx_labels(G, pos);

print(poly.get_any_border())

In [ ]:
import numpy as np
import collections

class EuclideanVertex2D(Vertex):
    def __init__(self, pos, any_outgoing=None):
        super(EuclideanVertex2D, self).__init__(any_outgoing)
        if not isinstance(pos, colledctions.Sized):
            raise ValueError(f"position must be Sized. Got {pos}.")
        if len(pos) != 2:
            raise ValueError(f"Got position of length {len(pos)} != 2.")
        self.pos = np.array(pos, dtype=np.float32)
    
    @property
    def x(self):
        return self.pos[0]
    
    @property
    def y(self):
        return self.pos[1]

In [ ]:
v1, v2 = Vertex(), Vertex()
h1, h2 = HalfEdge(orig=v1, dest=v2), HalfEdge(orig=v2, dest=v1)
h1.rev = h2
h2.rev = h1
e = Edge(h1, h2)

In [ ]:
e[h1.orig]

In [ ]:
from copy import copy

v = Vertex()
d = dict()
d[v] = 3
v.any_outgoing = 123
d[v]